Spectral
problem 1

In [11]:
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh

def read_graph(file_path):
    with open(file_path, 'r') as file:
        next(file)  # Skip the first line
        edges = [tuple(map(int, line.strip().split())) for line in file]
    # Finding the maximum node ID to construct the adjacency matrix (base on that the ID are numbers)
    max_node = max(max(edge) for edge in edges)
    # Creating sparse adjacency matrix
    row, col = zip(*edges)
    data = np.ones(len(edges))
    adj_matrix = csr_matrix((data, (row, col)), shape=(max_node + 1, max_node + 1))
    return adj_matrix

def compute_modularity_matrix(adj_matrix):
    n = adj_matrix.shape[0]
    degrees = np.array(adj_matrix.sum(axis=1)).flatten()
    total_edges = degrees.sum()
    if total_edges == 0:
        print("Warning: Total edges are zero. Returning a zero modularity matrix.")
        return csr_matrix((n, n))
    modularity_matrix = adj_matrix - csr_matrix(np.outer(degrees, degrees) / total_edges)
    return modularity_matrix

def partition_into_two_clusters(adj_matrix, global_node_indices, results=None):
    if results is None:
        results = {}
    n = adj_matrix.shape[0]
    if n <= 1:  # If the subgraph has at most one node, it will assign it to a single cluster
        for i in range(n):
            results[global_node_indices[i]] = 0
        return results
    modularity_matrix = compute_modularity_matrix(adj_matrix)
    # Computing the leading eigenvector using scipy.sparse.linalg.eigsh for sparse matrices
    eigenvalues, eigenvectors = eigsh(modularity_matrix, k=1, which='LA')
    leading_eigenvector = eigenvectors[:, 0]
    print("Leading eigenvector (top 10 values):", leading_eigenvector[:10])
    # Partitioning the nodes based on the sign of the eigenvector
    cluster_a = np.where(leading_eigenvector > 0)[0]
    cluster_b = np.where(leading_eigenvector <= 0)[0]
    # Assigning nodes to out two clusters
    for i in cluster_a:
        results[global_node_indices[i]] = 0
    for i in cluster_b:
        results[global_node_indices[i]] = 1
    return results

def write_results(results, output_file):
    cluster_counts = {}
    with open(output_file, 'w') as file:
        for node, cluster in sorted(results.items()):
            file.write(f"id[{node}] -> {cluster}\n")
            if cluster not in cluster_counts:
                cluster_counts[cluster] = 0
            cluster_counts[cluster] += 1
    print("\nCluster Counts:")
    for cluster, count in sorted(cluster_counts.items()):
        print(f"Cluster {cluster}: {count} nodes")

file_path = "F:/Aidin/Universities/Amirkabir (Master)/Lessons/Term 2/Analysis of Complex Networks/Assignments/HW2/Answer/Dataset/Spectral/data3.txt"
output_file = "F:/Aidin/Universities/Amirkabir (Master)/Lessons/Term 2/Analysis of Complex Networks/Assignments/HW2/Answer/Dataset/Spectral/clusters_output.txt"
adj_matrix = read_graph(file_path)
if adj_matrix.shape[0] != 8298:
    print(f"Error: The number of nodes ({adj_matrix.shape[0]}) does not match the expected 8298 nodes.")
else:
    global_node_indices = np.arange(adj_matrix.shape[0])
    results = partition_into_two_clusters(adj_matrix, global_node_indices)
    write_results(results, output_file)

Leading eigenvector (top 10 values): [ 6.94760583e-05 -2.52414298e-04 -1.38963612e-04  8.02222073e-03
  1.49196457e-02  1.21348671e-02  1.34927142e-01  1.47217865e-02
  6.87023341e-02  2.68233931e-02]

Cluster Counts:
Cluster 0: 4026 nodes
Cluster 1: 4272 nodes
